# Setup môi trường

In [39]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [40]:
# Set cố định random seed
np.random.seed(42)
tf.random.set_seed(42)

# Tìm hiểu dữ liệu

In [41]:
df = pd.read_csv("train_data.csv")
df = df.sample(
    n=100000,
    random_state=42
)
df.head(5)

,sentence,sentiment
704983,me too i think nanny should make me dinner now,0
1081081,i sent comments of love to her l,1
399206,dah they aren t on mine either my twitpic isn ...,0
414974,omg i need sleep but i work at,0
402589,im sorry im such a loser but only really ace p...,0


# Kiểm tra dữ liệu

In [42]:
df.info()
df['sentiment'].value_counts()
df = df.dropna()
X = df['sentence']
y = df['sentiment']

<class 'pandas.DataFrame'>
Index: 100000 entries, 704983 to 69430
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   sentence   100000 non-null  str  
 1   sentiment  100000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 8.5 MB


# Chia train/test

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [44]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [45]:
# Số lượng từ tối đa và độ dài câu
vocab_size = 10000
max_length = 400
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train)
X_train_seqs = tokenizer.texts_to_sequences(X_train)

X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_train_padded = pad_sequences(
    X_train_seqs,
    maxlen=max_length,
    padding='post',
    truncating='post'
)


X_test_padded = pad_sequences(
    X_test_seqs,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Train:", len(X_train_padded))

print("Test:", len(X_test_padded))

Train: 80000
Test: 20000


In [46]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


model = Sequential([
    Embedding(
        input_dim=10000,
        output_dim=64,
        input_length=400
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        1,
        activation='sigmoid'
    )
])

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [47]:
from tensorflow.keras.callbacks import ModelCheckpoint


checkpoint = ModelCheckpoint(
    filepath='model_epoch_{epoch:02d}.keras',
    save_weights_only=False,
    save_best_only=False,
    monitor='val_loss',
    verbose=1
)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
history = model.fit(
    X_train_padded,
    y_train,
    validation_data=(X_test_padded, y_test),
    epochs=5,
    callbacks=[checkpoint]
)

Epoch 1/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.4992 - loss: 0.6936
Epoch 1: saving model to model_epoch_01.keras

Epoch 1: finished saving model to model_epoch_01.keras
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 321s 127ms/step - accuracy: 0.4994 - loss: 0.6935 - val_accuracy: 0.5032 - val_loss: 0.6931
Epoch 2/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.4956 - loss: 0.6933
Epoch 2: saving model to model_epoch_02.keras

Epoch 2: finished saving model to model_epoch_02.keras
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 295s 118ms/step - accuracy: 0.4983 - loss: 0.6932 - val_accuracy: 0.5032 - val_loss: 0.6931
Epoch 3/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.5014 - loss: 0.6934
Epoch 3: saving model to model_epoch_03.keras

Epoch 3: finished saving model to model_epoch_03.keras
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 292s 117ms/step - accuracy: 0.5000 - loss: 0.6933 - val_accuracy: 0.5032 - val_loss: 0.6931
Epoch 4/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - a

# Lưu mô hình

In [48]:
model.save('tweet_sentiment_model.keras')

# Sử dụng mô hình

In [49]:
from tensorflow.keras.models import load_model

loaded_model = load_model("model_epoch_01.keras")

test_reviews = [
    "lebron and zydrunas are such an awesome duo",
    "lebron is a beast nobody in the nba comes even close",
    "downloading apps for my iphone so much fun there literally is an app for just about anything",
    "history exam studying ugh",
    "good news just had a call from the visa office saying everything is fine what a relief i am sick of scams out there stealing",
    "i hate revision it s so boring i am totally unprepared for my exam tomorrow things are not looking good",
    "awesome come back from via",
    "oh yes but if gm dies it will only be worth more boo hahaha",
]
test_gt = [1,1,0,0,1,0,1,0]
test_seqs = tokenizer.texts_to_sequences(test_reviews)
test_padded = pad_sequences(
    test_seqs,
    maxlen=max_length,
    padding='post',
    truncating='post'
)
preds = loaded_model.predict(test_padded)
preds

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step


array([[0.49574643],
       [0.49574643],
       [0.49574643],
       [0.49574643],
       [0.49574643],
       [0.49574643],
       [0.49574643],
       [0.49574643]], dtype=float32)

In [50]:
classes = {
    1: "positive",
    0: "negative"
}
threshold = 0.5
pred_scores = np.asarray(preds).reshape(-1)

acc = 0
for i in range(len(test_reviews)):
    review = test_reviews[i]
    score = float(pred_scores[i])
    true_label = test_gt[i]
    prediction = int(score > threshold)

    print(
        review,
        "| True:",
        classes[true_label],
        "| Predict:",
        classes[prediction],
        "| Score:",
        round(score,3)
    )

    if prediction == true_label:
        acc += 1


print(
    "\nAccuracy:",
    round(acc / len(test_reviews),2)
)

lebron and zydrunas are such an awesome duo | True: positive | Predict: negative | Score: 0.496
lebron is a beast nobody in the nba comes even close | True: positive | Predict: negative | Score: 0.496
downloading apps for my iphone so much fun there literally is an app for just about anything | True: negative | Predict: negative | Score: 0.496
history exam studying ugh | True: negative | Predict: negative | Score: 0.496
good news just had a call from the visa office saying everything is fine what a relief i am sick of scams out there stealing | True: positive | Predict: negative | Score: 0.496
i hate revision it s so boring i am totally unprepared for my exam tomorrow things are not looking good | True: negative | Predict: negative | Score: 0.496
awesome come back from via | True: positive | Predict: negative | Score: 0.496
oh yes but if gm dies it will only be worth more boo hahaha | True: negative | Predict: negative | Score: 0.496

Accuracy: 0.5
